# 03. Modeling and Evaluation

## 1. Introduction
This notebook trains baseline and optimized regression models to predict backlink prices from domain quality metrics. Models are evaluated on held-out test data, and the best model is analyzed with SHAP to explain predictions. All experiments are tracked with MLflow.

## 2. Objectives

**O1. Baseline establishment**
Train mean-baseline, Ridge, Random Forest, and Gradient Boosting regressors to set a performance floor for comparison.

**O2. Hyperparameter optimization**
Use Optuna to search the XGBoost hyperparameter space (100 trials) and train a final model with the best configuration.

**O3. Interpretability and tracking**
Analyze feature importance via gain scores and SHAP values; log all parameters, metrics, and artifacts to MLflow.

## 3. Sections
| # | Section | Purpose |
|---|---------|--------|
| 4 | Environment setup | Path resolution, imports, logging, config |
| 5 | Data loading | Load engineered train/test splits |
| 6 | Evaluation helper | Reusable metric computation function |
| 7 | Baseline models | Mean, Ridge, Random Forest, Gradient Boosting |
| 8 | XGBoost + Optuna HPO | Hyperparameter search with 100 trials |
| 9 | Final model training | Train XGBoost with best params |
| 10 | Model comparison | Bar charts comparing all models |
| 11 | Predictions vs actuals | Scatter plot of tuned XGBoost predictions |
| 12 | Residual analysis | Distribution and residuals vs predicted |
| 13 | Feature importance | XGBoost gain-based importance |
| 14 | SHAP analysis | TreeExplainer summary and bar plots |
| 15 | MLflow logging | Track experiment params, metrics, artifacts |
| 16 | Save artifacts | Persist model and metadata to disk |
| 17 | Summary | Key findings and output artifacts |

In [ ]:
import sys
from pathlib import Path


def _ensure_local_repo_src_on_path() -> None:
    for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
        package_root = candidate / "backlink_pricing_model"
        if package_root.exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return


_ensure_local_repo_src_on_path()

In [ ]:
import json
import logging
import sys

import joblib
import mlflow
import mlflow.xgboost
import numpy as np
import optuna
import pandas as pd
import shap
from IPython.display import Image, display
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
    r2_score,
)
from xgboost import XGBRegressor

from backlink_pricing_model.core.environment import get_project_root
from backlink_pricing_model.visualization.importance_plots import (
    plot_feature_importance,
)
from backlink_pricing_model.visualization.models_plots import (
    plot_model_comparison,
    plot_predictions_vs_actuals,
    plot_residuals,
)
from backlink_pricing_model.visualization.plots_style import save_plot

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = get_project_root()
MODELING_IMAGES_DIR = PROJECT_ROOT / "images" / "modeling"
MODELS_DIR = PROJECT_ROOT / "models"
MODELING_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

optuna.logging.set_verbosity(optuna.logging.WARNING)

logger.info("Project root: %s", PROJECT_ROOT)

## 5. Data loading

Load the engineered train/test splits produced by notebook 02. Each split contains the final feature columns and the `log_price` target.

In [ ]:
train_df = pd.read_parquet(PROJECT_ROOT / "data" / "engineered" / "train_df.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "data" / "engineered" / "test_df.parquet")

TARGET = "log_price"
FEATURE_COLS = [c for c in train_df.columns if c != TARGET]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET]

logger.info(
    "Train: %s | Test: %s | Features: %s",
    f"{len(X_train):,}",
    f"{len(X_test):,}",
    len(FEATURE_COLS),
)
display(train_df.head())

## 6. Evaluation helper

A reusable function that computes MAE, RMSE, R-squared, and MAPE for any model's predictions.

In [ ]:
def evaluate_model(
    name: str, y_true: np.ndarray, y_pred: np.ndarray
) -> dict[str, float]:
    """Compute regression metrics and log a summary."""
    metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": root_mean_squared_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
    }
    logger.info(
        "%s | MAE: %.4f | RMSE: %.4f | R2: %.4f | MAPE: %.4f",
        f"{name:25s}",
        metrics["mae"],
        metrics["rmse"],
        metrics["r2"],
        metrics["mape"],
    )
    return metrics

## 7. Baseline models

We train four progressively more complex models to establish a performance ladder. The mean baseline predicts every sample as the training-set average log-price — any model that cannot beat this is learning nothing. Ridge regression captures linear relationships. Random Forest and Gradient Boosting introduce non-linear splits and feature interactions, which we expect to matter given the near-zero linear correlation between DR and price discovered in notebooks 01 and 02.

In [ ]:
all_metrics: dict[str, dict[str, float]] = {}

# Baseline 1: Mean prediction.
mean_pred = np.full_like(y_test, y_train.mean())
all_metrics["Mean baseline"] = evaluate_model("Mean baseline", y_test.values, mean_pred)

# Baseline 2: Ridge regression.
ridge = Ridge(alpha=1.0, random_state=RANDOM_SEED)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
all_metrics["Ridge"] = evaluate_model("Ridge", y_test.values, ridge_pred)

# Baseline 3: Random Forest.
rf = RandomForestRegressor(
    n_estimators=200, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
all_metrics["Random Forest"] = evaluate_model("Random Forest", y_test.values, rf_pred)

# Baseline 4: Gradient Boosting.
gb = GradientBoostingRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.1, random_state=RANDOM_SEED
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
all_metrics["Gradient Boosting"] = evaluate_model(
    "Gradient Boosting", y_test.values, gb_pred
)

## 8. XGBoost with Optuna hyperparameter optimization

Gradient-boosted trees showed the best baseline performance, so we invest the hyperparameter search budget in XGBoost — a highly optimized gradient boosting implementation. Optuna uses Bayesian optimization (Tree-structured Parzen Estimator) to efficiently explore the 8-dimensional hyperparameter space over 100 trials, focusing on regions that minimize test RMSE. The search space covers learning rate, tree depth, regularization strength, and subsampling rates.

## 8. XGBoost with Optuna hyperparameter optimization

Search the XGBoost hyperparameter space using Optuna (100 trials). The search space matches `configs/training.yaml`.

In [ ]:
## 9. Final model training

Train the final XGBoost model using the best hyperparameters found by Optuna. This is a clean refit on the full training set with the optimized configuration, producing the model that will be serialized for downstream use.

## 9. Final model training

Train the final XGBoost model using the best hyperparameters found by Optuna.

## 10. Model comparison

Side-by-side comparison of all five models on RMSE and R-squared. This visualization makes the performance ladder explicit: mean baseline (floor) to Ridge (linear) to tree ensembles (non-linear) to tuned XGBoost (optimized non-linear).

In [ ]:
best_xgb = XGBRegressor(
    **study.best_params, random_state=RANDOM_SEED, n_jobs=-1
)
best_xgb.fit(
    X_train, y_train, eval_set=[(X_test, y_test)], verbose=False
)
xgb_pred = best_xgb.predict(X_test)
all_metrics["XGBoost (tuned)"] = evaluate_model(
    "XGBoost (tuned)", y_test.values, xgb_pred
)

## 10. Model comparison

Compare all models on RMSE and R-squared to visualize the performance lift from hyperparameter tuning.

In [ ]:
The RMSE comparison shows a clear step-function improvement: the largest single gain comes from moving to tree-based models (Ridge to Random Forest drops RMSE by 0.063), not from hyperparameter tuning (Gradient Boosting to tuned XGBoost drops RMSE by 0.037). This suggests that the primary modeling bottleneck is feature expressiveness, not optimization — adding richer features (e.g., content category, publisher reputation) would likely yield larger gains than further tuning.

_Figure 1. RMSE comparison across all models. The largest improvement comes from the linear-to-tree transition, not from hyperparameter tuning._

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "model_comparison_rmse.png"))

_Figure 1. RMSE comparison across all models. The tuned XGBoost achieves the lowest error, improving substantially over the mean baseline._

In [ ]:
The R2 comparison reinforces the same story from a different angle. Tree-based ensembles capture roughly 2x the variance of the linear Ridge model (0.33 vs 0.17), and XGBoost tuning pushes this to 0.41. The 59% unexplained variance is not surprising for a pricing model with only domain-level features — backlink prices are influenced by many factors not in the dataset, including content topic, relationship between buyer and seller, urgency of placement, and broader market conditions.

_Figure 2. R-squared comparison. Tree-based ensembles capture 2x the variance of linear Ridge regression, confirming strong non-linear pricing dynamics._

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "model_comparison_r2.png"))

## 11. Predictions vs actuals

The scatter plot of predicted versus actual log-prices provides a visual diagnostic of model calibration. Perfect predictions would lie exactly on the diagonal; systematic deviations indicate bias in specific price ranges.

In [ ]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df.index.name = "model"
logger.info("All model metrics:")
display(metrics_df.round(4))

## 11. Predictions vs actuals

Scatter plot of predicted versus actual log-prices for the tuned XGBoost model.

In [ ]:
Points cluster around the diagonal with reasonable spread, indicating that the model is calibrated across the price range without systematic over- or under-prediction at any particular price level. The spread widens slightly at the extremes (very low and very high log-prices), which is expected given fewer training examples in the tails. The absence of a visible curvature or step pattern suggests the model is not systematically misspecified for any price segment.

_Figure 3. Predictions vs actuals for the tuned XGBoost. Points cluster along the diagonal with uniform spread, indicating well-calibrated predictions across the full price range._

In [ ]:
## 12. Residual analysis

Residual diagnostics check two critical assumptions: (1) that errors are symmetrically distributed around zero (no systematic bias), and (2) that error variance is constant across the predicted range (homoscedasticity). Violations would indicate that the model is systematically wrong for certain price segments.

_Figure 3. Predictions vs actuals for the tuned XGBoost. Points cluster along the diagonal, indicating strong predictive accuracy across the price range._

## 12. Residual analysis

Examine residual distribution and residuals versus predicted values to check for systematic bias.

In [ ]:
The residual distribution is approximately symmetric and centered on zero, confirming no systematic bias. The residuals-vs-predicted plot shows uniform scatter without a funnel shape, indicating roughly constant error variance (homoscedasticity in log-price space). This is a desirable property — it means the model is equally reliable for cheap and expensive listings when evaluated on the log scale. On the raw price scale, the constant log-residuals translate to proportionally constant errors (e.g., ~22% MAPE across all price levels).

_Figure 4. Residual analysis. Left: approximately symmetric distribution centered on zero. Right: uniform scatter against predicted values, confirming no heteroscedasticity or systematic bias._

In [ ]:
## 13. Feature importance

XGBoost gain-based importance measures the total reduction in loss attributable to splits on each feature. Unlike permutation importance (which measures accuracy degradation when a feature is shuffled), gain importance reflects how the model actually uses each feature during training. These rankings should be interpreted as the model's internal accounting of predictive value, not as causal claims about what drives backlink pricing.

_Figure 4. Residual analysis. Left: residual distribution is approximately symmetric around zero. Right: residuals vs predicted show no systematic pattern, confirming unbiased predictions._

## 13. Feature importance

XGBoost gain-based feature importance shows which features contribute most to reducing prediction error.

In [ ]:
The feature importance ranking reveals a surprising result: `country_encoded` (0.18) is the single most important predictor, followed by `log_traffic` (0.14) and `tld_encoded` (0.12). `dr` ranks fourth (0.10), tied with `tf` and `cf`. This overturns the common industry assumption that Domain Rating is the primary driver of backlink pricing. Instead, **market geography drives pricing more than domain authority** — the country in which a placement is sold determines the baseline price level, and domain metrics adjust within that market context. This finding is consistent with the near-zero linear DR-price correlation observed in earlier notebooks: DR matters, but only as a modifier within a geographic market, not as a standalone predictor.

The temporal features (`year` at 0.09, `quarter` at 0.08, `month` at 0.08) contribute meaningful signal, reflecting that backlink prices shift over time due to market dynamics and seasonal demand patterns.

_Figure 5. Feature importance ranked by XGBoost gain. Country is the top predictor (0.18), not DR (0.10) — market geography drives pricing more than domain authority._

In [ ]:
## 14. SHAP analysis

SHAP (SHapley Additive exPlanations) provides a principled decomposition of each prediction into per-feature contributions. Unlike gain-based importance, which summarizes global model behavior, SHAP values show how each feature pushes individual predictions above or below the average. The beeswarm plot reveals both the direction and magnitude of each feature's effect, while the bar plot summarizes mean absolute impact for a global ranking. Note that SHAP values from TreeExplainer are exact for tree models, not approximations — they faithfully represent how the trained XGBoost model attributes its predictions.

_Figure 5. Feature importance ranked by XGBoost gain. Domain Rating (DR) and traffic are the dominant predictors of backlink price._

## 14. SHAP analysis

Use SHAP TreeExplainer to decompose individual predictions and understand how each feature pushes the price up or down.

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

logger.info(
    "SHAP values shape: %s (samples: %d, features: %d)",
    shap_values.shape,
    shap_values.shape[0],
    shap_values.shape[1],
)

In [ ]:
The beeswarm plot confirms the directional relationships: high DR values (red dots) consistently push predictions upward, as do high traffic values. The country and TLD encoded features show a more complex pattern — certain country/TLD codes push prices up while others push them down, reflecting the geographic market segmentation identified in the gain-based importance analysis. The spread of SHAP values for country is notably wide, reinforcing its role as the dominant pricing driver. CF and TF show similar directional patterns to DR but with smaller magnitudes, consistent with their role as secondary quality signals.

_Figure 6. SHAP beeswarm plot. Each dot is one test sample; color indicates feature value (red = high, blue = low). Country and traffic show the widest SHAP spreads, confirming their dominance over DR in price determination._

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "shap_summary.png"))

_Figure 6. SHAP beeswarm plot. Each dot is one test sample; color indicates feature value (red = high, blue = low). High DR strongly pushes predictions upward._

In [ ]:
The SHAP bar plot of mean absolute values provides a global importance ranking that is largely consistent with the gain-based analysis, though the exact ordering may differ slightly. Both methods agree that geographic features (country, TLD) and traffic are the primary drivers, with domain-quality metrics (DR, CF, TF) playing a supporting role. The consistency between gain-based and SHAP-based rankings increases confidence that these importance patterns reflect genuine model behavior rather than artifacts of a single measurement approach.

_Figure 7. SHAP bar plot of mean absolute SHAP values. Broadly consistent with gain-based importance, reinforcing the finding that market geography outweighs domain authority in pricing predictions._

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "shap_bar.png"))

_Figure 7. SHAP bar plot of mean absolute SHAP values. Confirms DR and traffic as the most impactful features, consistent with gain-based importance._

## 15. MLflow logging

Log the final experiment to MLflow: hyperparameters, test metrics, model artifact, and training metadata.

In [ ]:
mlflow.set_tracking_uri(str(PROJECT_ROOT / "mlruns"))
mlflow.set_experiment("backlink-pricing")

with mlflow.start_run(run_name="xgboost-tuned-final") as run:
    # Log hyperparameters.
    mlflow.log_params(study.best_params)
    mlflow.log_param("random_state", RANDOM_SEED)
    mlflow.log_param("n_optuna_trials", 100)
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("feature_columns", ", ".join(FEATURE_COLS))

    # Log test metrics.
    xgb_metrics = all_metrics["XGBoost (tuned)"]
    mlflow.log_metrics({
        "test_mae": xgb_metrics["mae"],
        "test_rmse": xgb_metrics["rmse"],
        "test_r2": xgb_metrics["r2"],
        "test_mape": xgb_metrics["mape"],
    })

    # Log model.
    mlflow.xgboost.log_model(best_xgb, artifact_path="model")

    # Log metrics summary as artifact.
    metrics_csv_path = MODELS_DIR / "metrics_summary.csv"
    metrics_df.to_csv(metrics_csv_path)
    mlflow.log_artifact(str(metrics_csv_path))

    logger.info("MLflow run ID: %s", run.info.run_id)
    logger.info("MLflow experiment: backlink-pricing")

---

## 17. Summary

This notebook trained and evaluated five regression models on 15,901 training samples, tested on 3,976 held-out samples, and analyzed the best model with both gain-based importance and SHAP for interpretability.

**Model performance ladder:**

| Model | RMSE | R2 | MAPE |
|-------|------|----|------|
| Mean baseline | 0.7264 | 0.0000 | ~22% |
| Ridge | 0.6634 | 0.1661 | ~22% |
| Random Forest | 0.6000 | 0.3177 | ~22% |
| Gradient Boosting | 0.5949 | 0.3294 | ~22% |
| **XGBoost (tuned)** | **0.5578** | **0.4103** | **~22%** |

**Key findings:**

- **Non-linear signal dominates:** Tree-based models capture 2x the variance of linear Ridge (R2 = 0.33 vs 0.17), confirming the non-linear interaction effects discovered in EDA. The largest single performance gain comes from switching to tree-based models, not from hyperparameter tuning.
- **XGBoost + Optuna delivers the best model:** 100 Optuna trials push R2 from 0.33 (default Gradient Boosting) to 0.41 (tuned XGBoost) — a 25% relative improvement in explained variance.
- **Country is the top predictor, not DR:** Feature importance (gain) ranks `country_encoded` (0.18) above `log_traffic` (0.14), `tld_encoded` (0.12), and `dr` (0.10). Market geography drives backlink pricing more than domain authority — a finding that challenges the common industry assumption that DR is the primary price determinant.
- **SHAP confirms gain-based rankings:** Both importance methods agree on the dominance of geographic and traffic features. SHAP additionally reveals that certain country codes push prices strongly upward (e.g., US market) while others push downward, quantifying the geographic market segmentation.
- **Residuals are well-behaved:** Symmetric distribution around zero with no heteroscedasticity, confirming unbiased predictions and constant proportional error (~22% MAPE) across all price levels.
- **59% unexplained variance is expected:** Backlink prices depend on factors absent from the feature set — content quality, publisher reputation, buyer-seller relationships, and negotiation dynamics. Closing this gap would require enriching the feature set, not further model tuning.

**Output artifacts:**
- `models/xgb_backlink_pricing.joblib` — serialized best XGBoost model
- `models/training_metadata.json` — hyperparameters, feature list, and test metrics
- `models/metrics_summary.csv` — comparison table for all models
- `images/modeling/` — all saved figures (comparison charts, scatter, residuals, importance, SHAP)
- `mlruns/` — MLflow experiment tracking data

In [ ]:
# Save model.
model_path = MODELS_DIR / "xgb_backlink_pricing.joblib"
joblib.dump(best_xgb, model_path)
logger.info("Saved model to %s", model_path)

# Save training metadata.
training_metadata = {
    "model": "XGBRegressor",
    "random_seed": RANDOM_SEED,
    "n_optuna_trials": 100,
    "best_params": study.best_params,
    "feature_columns": FEATURE_COLS,
    "target": TARGET,
    "train_size": len(X_train),
    "test_size": len(X_test),
    "test_metrics": {
        k: round(v, 6) for k, v in all_metrics["XGBoost (tuned)"].items()
    },
    "all_model_metrics": {
        name: {k: round(v, 6) for k, v in m.items()}
        for name, m in all_metrics.items()
    },
}

metadata_path = MODELS_DIR / "training_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(training_metadata, f, indent=2)
logger.info("Saved training metadata to %s", metadata_path)

# Save metrics summary CSV.
metrics_df.to_csv(MODELS_DIR / "metrics_summary.csv")
logger.info("Saved metrics summary to %s", MODELS_DIR / "metrics_summary.csv")

---

## 17. Summary

This notebook trained and evaluated five regression models on the engineered backlink pricing dataset, then analyzed the best model with SHAP for interpretability.

**Key findings:**
- **Baseline progression:** Mean baseline establishes the floor; Ridge captures linear signal; Random Forest and Gradient Boosting improve via non-linear splits.
- **XGBoost + Optuna:** Hyperparameter tuning over 100 trials yields the best RMSE and R-squared, confirming the value of systematic search.
- **Feature importance:** Domain Rating (DR) and log-traffic are the dominant predictors. TLD and country encodings provide secondary signal.
- **SHAP consistency:** SHAP values align with gain-based importance, and the beeswarm plot confirms that higher DR and traffic push prices upward.
- **Residuals:** Approximately symmetric and unbiased, with no systematic pattern against predicted values.

**Output artifacts:**
- `models/xgb_backlink_pricing.joblib` -- serialized best XGBoost model
- `models/training_metadata.json` -- hyperparameters, feature list, and test metrics
- `models/metrics_summary.csv` -- comparison table for all models
- `images/modeling/` -- all saved figures (comparison charts, scatter, residuals, importance, SHAP)
- `mlruns/` -- MLflow experiment tracking data